In [4]:
import spacy
from spacy.pipeline import EntityRuler
from spacy.matcher import Matcher
from spacy import displacy
from spacy.pipeline import EntityLinker
from scispacy.abbreviation import AbbreviationDetector
#from spacy.kb import InMemoryLookupKB
#!python -m spacy download en_core_web_lg

import os
from pathlib import Path
#from pypdf import PdfReader as pr
import json
import random

In [9]:
with open('list_files\\genus_names.json', 'r', encoding='utf-8') as f:
    genera = json.load(f)
    
with open('list_files\\species_names.json', 'r', encoding='utf-8') as f:
    species = json.load(f)
    
with open('list_files\\utilizations_list.json', 'r', encoding='utf-8') as f:
    utilizations = json.load(f)
    
utilization_patterns = []
for utilize in utilizations:
    utilization_patterns.append({'label': 'KIND_OF_UTILIZATION_TESTED', 'pattern': utilize})

with open('list_files\\genus_endings.json', 'r', encoding='utf-8') as f:
    genera_endings = json.load(f)

with open('list_files\\species_6_endings.json', 'r', encoding='utf-8') as f:
    species_endings = json.load(f)
    
with open('list_files\\species_6_beginnings.json', 'r', encoding='utf-8') as f:
    species_beginnings = json.load(f)
    
species_beginnings = species_beginnings + [spec.title() for spec in species_beginnings]

print(species_beginnings[:20])
print(species_beginnings[-20:])
    
#with open('list_files\\microbe_patterns.json', 'r', encoding='utf-8') as f:
#    microbes_patterns = json.load(f)

['rhodoc', 'globos', 'jiangx', 'emoden', 'cauliu', 'subhya', 'kakico', 'friede', 'invisu', 'huakui', 'halste', 'turoni', 'lunulo', 'sharpe', 'orinoc', 'adhaer', 'granos', 'reprae', 'monoic', 'peruvi']
['Murale', 'Seroti', 'Cherno', 'Navarr', 'Zobell', 'Codiae', 'Bryoph', 'Monume', 'Dagmat', 'Distyl', 'Pulla', 'Lylae', 'Pudica', 'Macera', 'Burtii', 'Cinnab', 'Sardoa', 'Sedime', 'Bowman', 'Gessar']


In [ ]:
#model = spacy.load('en_core_web_lg')

model = spacy.load('..\\dissertation DLC content\\en_ner_bionlp13cg_md-0.5.4\\en_ner_bionlp13cg_md\\en_ner_bionlp13cg_md-0.5.4')

custom_labels = ['MICROBE_NAME']

ner = model.get_pipe('ner')
for label in custom_labels:
    ner.add_label(label)

regular_vocab = list(model.vocab.strings)

print(len(regular_vocab))

regular_vocab = list(set(regular_vocab) - set(genera) - set(species) - set([gen.lower() for gen in genera]))

print(len(regular_vocab))

model.add_pipe('abbreviation_detector', last=True)

6296370
6279586


In [11]:
ruler = model.add_pipe('entity_ruler', before='ner')

""" def create_kb(vocab):
    kb = InMemoryLookupKB(vocab, entity_vector_length=128)
    return kb

linker = model.add_pipe('entityLinker', after='ner') """

#linker.set_kb(create_kb)

#linker = EntityLinker(model.vocab)
    
species_regex = f"(?=({'|'.join(species_beginnings)})[a-z]*)(?=[a-z]*({'|'.join(species_endings)})[\\.,]?)"

microbes_patterns = [
    #Xx?x?. species
    [{'TEXT': {'REGEX': '[A-Z][a-z]{0,2}'}, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': species_regex}}],
    #X. species
    [{'TEXT': {'REGEX': '[A-Z]{1}'}, 'IS_TITLE': True, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': species_regex}}],
    [{'TEXT': {'REGEX': '[A-Z]{1}\\.'}, 'IS_TITLE': True}, {'TEXT': {'REGEX': species_regex}}],
    #Xxx sp. species
    [{'TEXT': {'REGEX': '[A-Z][a-z]{0,2}'}, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': 'sp.'}, {'SHAPE': 'xxxx'}],
    #X. sp. species
    [{'TEXT': {'REGEX': '[A-Z]'}, 'IS_ALPHA': False, 'LENGTH': 2, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': 'sp.'}, {'TEXT': {'REGEX': '[A-Za-z]{4,}'}}],
    [{'TEXT': {'REGEX': '[A-Z]\\.'}, 'IS_ALPHA': False, 'LENGTH': 2}, {'TEXT': 'sp', 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': '[A-Za-z]{4,}'}}],
    #genus species (standard)
    [{'TEXT': {'IN': genera}, 'IS_TITLE': True, 'LENGTH': {'>=': 4}}, {'TEXT': {'REGEX': species_regex, 'NOT_IN': regular_vocab + ['species']}}],
    #genus spp. OR genus sp.
    [{'TEXT': {'IN': genera}, 'IS_TITLE': True, 'LENGTH': {'>=': 4}}, {'TEXT': {'REGEX': 'spp.'}}],
    [{'TEXT': {'IN': genera}, 'IS_TITLE': True, 'LENGTH': {'>=': 4}}, {'TEXT': {'REGEX': 'sp.'}}],
    #Candidatus species
    [{'TEXT': 'Candidatus', 'IS_TITLE': True}, {'SHAPE': 'xxxx'}],
    #candidatus genus species
    [{'TEXT': 'Candidatus', 'IS_TITLE': True}, {'SHAPE': 'Xxxx', 'IS_TITLE': True}, {'TEXT': {'REGEX': species_regex, 'NOT_IN': regular_vocab}}],
    #unidentified/uncultured genus species
    [{'TEXT': {'IN': ['unidentified', 'uncultured']}}, {'TEXT': {'REGEX': f"{'|'.join(genera)}"}, 'IS_TITLE': True}, {'SHAPE': 'xxxx'}],
    [{'TEXT': {'IN': ['unidentified', 'uncultured']}}, {'TEXT': {'REGEX': '[A-Za-z]+-?[A-z]*'}}, {'TEXT': 'bacteria'}],
    [{'TEXT': {'IN': ['unidentified', 'uncultured']}}, {'TEXT': {'REGEX': '[A-Za-z]+\\s?clone'}}, {'TEXT': 'REGEX'}]
]

"""     
    #genus species strain else
    [{'TEXT': {'IN': genera}, 'IS_TITLE': True, 'LENGTH': {'>=': 4}}, {'TEXT': {'REGEX': species_regex, 'NOT_IN': regular_vocab}}, {'TEXT': {'IN': ['strain']}}, {'TEXT': {'REGEX': '[A-z|0-9]{4,}'}}],
    #genus species var./subsp. else
    [{'TEXT': {'IN': genera}, 'IS_TITLE': True, 'LENGTH': {'>=': 4}}, {'TEXT': {'REGEX': species_regex, 'NOT_IN': regular_vocab}}, {'TEXT': {'IN': ['var', 'subsp', 'strain']}, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': '[A-z|0-9]{4,}'}}],
    [{'TEXT': {'IN': genera}, 'IS_TITLE': True, 'LENGTH': {'>=': 4}}, {'TEXT': {'REGEX': species_regex, 'NOT_IN': regular_vocab}}, {'TEXT': {'REGEX': '(var|subsp|strain)\\.'}}, {'TEXT': {'REGEX': '[A-z|0-9]{4,}'}}],
    #g. species var else
    [{'TEXT': {'REGEX': '[A-Z][a-z]{0,2}'}, 'IS_TITLE': True, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': species_regex, 'NOT_IN': regular_vocab}}, {'TEXT': {'REGEX': '(var|subsp|strain)\\.'}}, {'TEXT': {'REGEX': '[A-z|0-9]{4,}'}}],
    [{'TEXT': {'REGEX': '[A-Z][a-z]{0,2}\\.'}, 'IS_TITLE': True}, {'TEXT': {'REGEX': species_regex, 'NOT_IN': regular_vocab}}, {'TEXT': {'REGEX': '(var|subsp|strain)\\.'}}, {'TEXT': {'REGEX': '[A-z|0-9]{4,}'}}],
    [{'TEXT': {'REGEX': '[A-Z][a-z]{0,2}'}, 'IS_TITLE': True, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': species_regex, 'NOT_IN': regular_vocab}}, {'TEXT': {'REGEX': '(var|subsp|strain)'}, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': '[A-z|0-9]{4,}'}}],
    [{'TEXT': {'REGEX': '[A-Z][a-z]{0,2}\\.'}, 'IS_TITLE': True}, {'TEXT': {'REGEX': species_regex, 'NOT_IN': regular_vocab}}, {'TEXT': {'REGEX': '(var|subsp|strain)'}, 'SPACY': False}, {'TEXT': '.'},{'TEXT': {'REGEX': '[A-z|0-9]{4,}'}}]
] """

ruler.add_patterns([{'label': 'MICROBE_NAME', 'pattern': pattern} for pattern in microbes_patterns])

#ruler.add_patterns(utilization_patterns)

Preliminary test untrained, just gazeteering this

In [12]:
with open('list_files\\annotations_all_sentences.json', 'r', encoding='utf-8') as f:
    testing_data = json.load(f)['annotations']

# Model testing

In [ ]:
#%%script False

#scorer = Scorer()
#examples = []
i = 1
for text, annotations in testing_data:
    print(i)
    doc = model(text)
    ents_and_labels = [{'label': ent.label_, 'text': ent.text, 'start': ent.start, 'end': ent.end} for ent in doc.ents if ent.label_ == 'MICROBE_NAME']
    #print(ents_and_labels)
    
    for abr in doc._.abbreviations:
        print(abr)

    """ linked_entity_labels = []
    for lent in doc._.linkedEntities:
        linked_entity_labels.append(lent.get_label())
    print(linked_entity_labels) """

    #example = Example.from_dict(doc, {'entities': annotations})
    #examples.append(example)
    displacy.render(doc, style="ent", jupyter=True, minify=True)

    """ with open('list_files\\ner_triples.jsonl', 'a', encoding='utf-8') as f:
        json.dump((text, ents_and_labels), f)
        f.write('\n') """
    print()
    i += 1
    
#scorer.score(examples)

1 In addition to the main LAB metabolites (lactic and acetic acids), they are capable to excrete other inhibition substances (ethanol, ammonia, diacetyl, acetoin, acetaldehyde, hydrogen peroxide, benzoate, formic acid, bacteriolytic enzymes, 2,3-butanediol, free fatty acids, bacteriocins, etc.


c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any pattern


2 mixed fermentation using L. thermotolerans and S. cerevisiae, where an increase in glycerol and 2-phenyl ethanol production

3 Growth rate of Candida utilis in the presence of various concentrations of lignosulphonic acids during the fermentation of xylose

4 oxidation of ethanol to acetic acid

5 oxidation of glucose to gluconic acid

6 The consumption of fructose by C. zemplinina and the consequent osmotic pressure release promotes a reduction in acetic acid production


c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)



7 Glucose and xylose were used in this study

8 those products are some amino acids—such as glutamate, lysine, and methionine—and SCP

9 Specifically, Lpb. plantarum strains co-metabolized with B. subtilis var. natto to elevate the content of pyrazines, characteristic VOCs in natto

10 We assessed the extent of soybean protein hydrolysis by measuring TCA-soluble peptide and amino acid nitrogen content


...

c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any pattern


233 The coffee pulp is mixed with a solution of urea, ammonium sulphate and potassium phosphate, having a final humidity of 60% and pH 5.5

234 it was decided to use pectin as substrate, since polygalacturonic acid become unsoluble at pH 3.0 and lower, while at pH 2.0, pectin is still soluble

235 Phenol in coffee pulp causes a sour and astringent taste in processed products, so a fermentation process is needed to degrade phenol compounds during aerobic fermentation

236 it was found that the longer the fermentation process lasts, the lower the reduced sugar content in the juice and sugar pulp of the coffee produced. This is because the sugar formed during fermentation is broken down by yeast to produce alcohol

237 the presence of LAB may provide benefits for yeast growth such as promoting or inhibiting various metabolic processes in yeast cells: metabolism of trehalose, ergosterol, certain amino acids


c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)
c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)



238 Citric acid and malic acid are naturally present in green coffee beans. The content of malic acid and citric acid can contribute to the citric and herbaceous taste after roasting [43]. In addition, malic acid can also provide apple flavor [44], but it did not appear in this study.

239 acetic acid can give a fruity taste when it is at low concentrations 


KeyboardInterrupt: 

In [18]:
additional_unannotated_sentences = [
    'Candidatus Bacterium thermotolerans or something idk',
    'fermentation is a very cool thing!'
]

for sent in additional_unannotated_sentences:
    print(sent)
    doc = model(sent)
    print([t.text for t in doc])
    ents_and_labels = list(zip([ent.label_ for ent in doc.ents], doc.ents))
    print(ents_and_labels)

Candidatus Bacterium thermotolerans or something idk
['Candidatus', 'Bacterium', 'thermotolerans', 'or', 'something', 'idk']
[]
fermentation is a very cool thing!
['fermentation', 'is', 'a', 'very', 'cool', 'thing', '!']
[]


# More training text

In [19]:
training_text = [
    ("The microbial consortium between Pediococcus acidilactici and Acetobacter cerevisiae enabled the metabolization of all the xylose contained in the liquor", 
    [(33, 57, "MICROBE"), (62, 84, "MICROBE")]),
    ("Some wild-type strains have a strong tolerance for lignocellulose-derived inhibitors, e.g. lactic acid-producing bacteria, Lactobacillus plantarum, have the ability to ferment in the presence of 8.0 g L^−1 furfural and 6.0 g L%−1 5-hydroxymethylfurfural (5-HMF)",
    [(123, 147, "MICROBE")]),
    ("Since A. cerevisiae is also a producer of acetic acid",
    [(6, 19, "MICROBE")]),
    ("the fermentation processes presented here also demonstrated the ability of P. acidilactici and A. cerevisiae, and co-cultivation, to consume different sugars",
    [(75, 90, "MICROBE"), (95, 109, "MICROBE")]),
    ("it can be seen that both P. acidilactici and A. cerevisiae were able to metabolize 5-HMF and furfural",
    [(25, 40, "MICROBE"), (45, 58, "MICROBE")]),
    ("Broad bean paste is usually made by the following three phases: a natural fermentation of chili-to-moromi, a conversion from broad bean to koji by Aspergillus oryzae",
    [(147, 165, "MICROBE")]),
    ("overnight grown RCM culture (Clostridium acetobutylicum) and incubated under anaerobic conditions for 24 h",
    [(29, 55, "MICROBE")]),
    ("Anaerobic fermentation without enzymatic assistance by Saccharomyces cerevisiae with temperature 30 °C was modeled in all scenarios", [(55, 79, "MICROBE")]),
    ("co-culture of Bacillus sp. MB2, Bacillus sp. WB8A and B. pumilus strain WB1A", [(14, 30, "MICROBE"), (32, 49, "MICROBE"), (54, 76, "MICROBE")]),
    ("co-cultures of Piromyces and Methanobrevibacter ruminantium", [(15, 24, "MICROBE"), (29, 59, "MICROBE")]),
    ("FA production from rice bran was four times greater than Aspergillus oryzae through fermentation of A. oryzae and Rhizopus oryzae co-culture.", [(57, 75, "MICROBE")]),
    ("lactic acid fermentation was performed with optimal OB hydrolysate by Lactobacillus delbrueckii subsp. bulgaricus OZZ4 and Lactobacillus plantarum OZH8",
    [(70, 118, "MICROBE"), (123, 151, "MICROBE")]),
    ("Strains of lactic acid bacteria (LAB); Lactobacillus delbrueckii subsp. bulgaricus OZZ4 and Lactobacillus plantarum OZH8",
    [(11, 37, "MICROBE"), (39, 87, "MICROBE"), (92, 120, "MICROBE")]),
    ("sorbitol is converted to sorbose by Gluconobacter oxydans",
    [(36, 57, "MICROBE")]),
    ("Pseudoglyconobacter Saccharoketogenes for 72 h to produce sodium keto-gluconic acid",
    [(0, 37, "MICROBE")]),
    ("The observation that lignosulphonic acids inhibit the fermentation of sucrose, glucose and invert sugar with Candida utilis",
    [(109, 123, "MICROBE")]),
    ("Saccharomyces cerevisiae ferments mainly hexoses",
    [(0, 24, "MICROBE")]),
    ("invertase produced by Candida utilis for degrading the disaccharide to glucose and fructose",
    [(22, 36, "MICROBE")]),
    ("B. subtilis fermented soybeans foods in various parts of the world. One of the common examples is Kinema",
    [(0, 11, "MICROBE")]),
    ("Bacillus spp. is the most dominant naturally fermenting agents in soybeans",
    [(0, 13, "MICROBE")]),
    ("B. subtilis is a group of related strains under the family Bacillaceae",
    [(0, 11, "MICROBE"), (59, 70, "MICROBE")]),
    ("These traditionally fermented soyfoods are littered with several other microbial species such as Enterococcus faecium, Candia parapsilosis, Geotrichum candidum etc.",
    [(97, 117, "MICROBE"), (119, 138, "MICROBE"), (140, 159, "MICROBE")]),
    ("prepared Thua-Nao using pure culture of B. subtilis",
    [(40, 51, "MICROBE")]),
    ("B. subtilis during Thua Nao fermentation releases proteinases that play important role in proteolysis of soy proteins",
    [(0, 11, "MICROBE")]),
    ("B. subtilis fermentation tends to increase total polyphenolic compound and anthocyanins, upto 10 and 250%, respectively during Natto fermentation of black soybeans",
    [(0, 11, "MICROBE")]),
    ("Fermentation of soybeans, particularly with B. subtilis",
    [(44, 55, "MICROBE")]),
    ("fermented soybeans inhibits adhesion of ETEC K88 in Rhizopus fermented soybeans",
    [(40, 48, "MICROBE"), (52, 60, "MICROBE")]),
    ("starter was prepared using yeast (Saccharomyces cerevisiae) and mold (Rhizopus oryzae)",
    [(34, 58, "MICROBE"), (70, 85, "MICROBE")]),
    ("Defined fermentation starter (prepared from rice using pure cultures of S. cerevisiae and R. oryzae) was mixed",
    [(72, 85, "MICROBE"), (90, 99, "MICROBE")]),
    ("similar alcohol content (6.68% v/v) in millet Jand fermented by using starter made from Rhizopus and Saccharomyces spp",
    [(88, 96, "MICROBE"), (101, 118, "MICROBE")]),
    ("semisolid state fermentation on glucose in taro fermentation using starter containing R. tankinensis, R. oryzae, R. chinensis and S. cerevisiae, but contrary to our finding, they reported a significantly higher yield of alcohol in semisolid fermentation than that of solid state fermentation",
    [(86, 100, "MICROBE"), (102, 111, "MICROBE"), (130, 143, "MICROBE")]),
    ("Spontaneous whey fermentation mostly depends on lactic acid bacteria (LAB), which convert lactose (the major component of whey) into lactic acid",
    [(48, 74, "MICROBE")]),
    ("lactic acid production of 5.3g/L after 48h at 37 ◦C using Lactobactillus bulgaricus for whey fermentation",
    [(58, 72, "MICROBE")]),
    ("L. acidophilus, L. bulgaricus, and L. casei presented more proteolytic activity than Str. thermophilus and Lb. paracasei",
    [(0, 15, "MICROBE"), (16, 29, "MICROBE"), (35, 43, "MICROBE"), (85, 102, "MICROBE"), (107, 120, "MICROBE")]),
    ("reuterin produced by Limosilactobacillus reuteri is reported as a potent compound with broad-range antimicrobial activity that inhibits fungi but also Gram-negative bacteria",
    [(21, 48, "MICROBE")]),
    ("Limosilactobacillus reuteri uses a CoA-dependent pathway",
    [(0, 27, "MICROBE")]),
    ("pediocins produced by Pediococcus spp.",
    [(22, 38, "MICROBE")]),
    ("Helveticin-M, produced by Lactobacillus crispatus",
    [(26, 49, "MICROBE")]),
    ("Lactococcus lactis and Lactiplantibacillus plantarum strains were studied in model cheeses against Listeria monocytogenes",
    [(0, 18, "MICROBE"), (23, 52, "MICROBE"), (99, 121, "MICROBE")]),
    ("Staph. equorum inhibited the growth of L. monocytogenes",
    [(0, 14, "MICROBE"), (39, 55, "MICROBE")]),
    ("C. crustorum, Lpb. plantarum and Lmb. fermentum decreased the levels of L. monocytogenes in cheese",
    [(0, 12, "MICROBE"), (14, 28, "MICROBE"), (33, 47, "MICROBE"), (72, 88, "MICROBE")]),
    ("E. faecium KE82 is suggested as a protective culture, but the indigenous bacteriocin-producing LAB might contribute to the inhibition of L. monocytogenes in Graviera",
    [(0, 10, "MICROBE"), (137, 153, "MICROBE")]),
    ("Pecorino Sardo PDO Lpb. plantarum (commercial) and an autochthonous LAB (Lb. delbruekii ssp. sunkii). Protection against L. monocytogenes Lb. delbruekii ssp. sunkii was as effective as the commercial culture for the protection against L. monocytogenes",
    [(19, 46, "MICROBE"), (73, 99, "MICROBE"), (121, 137, "MICROBE"), (138, 164, "MICROBE"), (235, 251, "MICROBE")]),
    ("Lcb. paracasei delayed the growth of Staph. aureus and L. monocytogenes in Coalho cheese",
    [(0, 14, "MICROBE"), (37, 50, "MICROBE"), (55, 71, "MICROBE")]),
    ("E. faecium CRL1879 ensured an efficient control of L. monocytogenes for up to 30 days without altering the organoleptic properties of the artisanal cheese",
    [(0, 18, "MICROBE"), (51, 67, "MICROBE")]),
    ("milk with a 10% yogurt starter containing Lactobacillus delbrueckii subsp. bulgaricus and Streptococcus thermophilus, also various S. cerevisiae concentrations",
    [(42, 85, "MICROBE"), (90, 116, "MICROBE"), (131, 144, "MICROBE")]),
    ("ethanol production by S. cerevisiae",
    [(22, 35, "MICROBE")]),
    ("S. cerevisiae addition was also found to slightly inhibit the growth of L. delbrueckii subsp. bulgaricus and S. thermophilus",
    [(0, 13, "MICROBE"), (72, 104, "MICROBE"), (109, 124, "MICROBE")]),
    ("Saccharomyces cerevisiae, a well-known yeast strain known for its health benefits, fermentative metabolism, and ethanol production capabilities, to enhance yogurt quality",
    [(0, 51, "MICROBE")]),
    ("it has been shown that S. cerevisiae could grow in milk and produce small amounts of ethanol, glycerol, and lactic acid",
    [(23, 36, "MICROBE")]),
    ("While milk itself is not a suitable growth medium for S. cerevisiae due to limitations like the inability to ferment milk’s lactose",
    [(54, 67, "MICROBE")]),
    ("L. delbrueckii subsp. bulgaricus and S. thermophilus can utilize lactose",
    [(0, 32, "MICROBE"), (37, 52, "MICROBE")]),
    ("However, L. delbrueckii subsp. bulgaricus and S. thermophilus cannot naturally use galactose",
    [(9, 41, "MICROBE"), (46, 61, "MICROBE")]),
    ("S. cerevisiae can metabolize galactose through its Leloir pathway producing UDP-glucose",
    [(0, 13, "MICROBE")]),
    ("S. cerevisiae culture was transferred and grown on Malt Extract Agar (MEA) slants at 30 °C",
    [(0, 13, "MICROBE")]),
    ("L. delbrueckii subsp. bulgaricus and S. thermophilus cultures were regrown on MRS agar slants at 42 °C for 2 days and on M17 agar at 37 °C for 2 days, respectively",
    [(0, 32, "MICROBE"), (37, 52, "MICROBE")]),
    ("30 °C to represent the suboptimal temperature for both L. delbrueckii subsp. bulgaricus and S. thermophilus and to provide the optimal temperature for S. cerevisiae",
    [(151, 164, "MICROBE")]),
    ("the treatment with S. cerevisiae addition (Figure 2b, Sc-1–Sc-4) showed a significantly lower pH than the milk without any culture (Figure 2b, Sc-0). In both experimental and control groups, increasing the concentration of S. cerevisiae resulted in a lower pH after incubation",
    [(19, 32, "MICROBE"), (223, 236, "MICROBE")]),
    ("indicating that S. cerevisiae could inhibit the growth of L. delbrueckii subsp. bulgaricus and S. thermophilus to some degree",
    [(16, 29, "MICROBE"), (58, 90, "MICROBE"), (95, 110, "MICROBE")]),
    ("L. delbrueckii subsp. bulgaricus and S. thermophilus support each other through protocooperation where metabolite exchange occurs",
    [(0, 32, "MICROBE"), (37, 52, "MICROBE")]),
    ("both L. delbrueckii subsp. bulgaricus and S. thermophilus have ethanol tolerance", [(5, 37, "MICROBE"), (42, 57, "MICROBE")]),
    ("here are some of my favourite letters of the alphabet: g, h, k, l, r, b and m!", []),
    ("I was once sent to the ER for endometriosis... they said I don't need antibiotics.", [])
]

# Other stuff

In [20]:
model.to_disk('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\custom_microbial_ner')

In [21]:
%%script False

optimizer = model.create_optimizer()

for i in range(20):
    print(i)
    random.shuffle(training_data)
    for text, annotations in training_data:
        doc = model.make_doc(text)
        annotations = [tuple(item) for item in annotations.get('entities')]
        example = Example.from_dict(doc, {'entities': annotations})
        model.update([example], sgd=optimizer)

Couldn't find program: 'False'


In [22]:
for file in os.listdir(os.getcwd() + "\\fermentation_papers"):
    filename = os.fsdecode(file)
    if filename.endswith('json'):
        with open(f"{os.getcwd()}\\fermentation_papers\\{filename}", 'r', encoding='utf-16') as f:
            text = json.load(f)
    """ elif filename.endswith('pdf'):
        path = Path(os.getcwd() + "\\fermentation_papers\\" + filename)
        reader = pr(path)
        text = ""
        for page in reader.pages:
            text = text + page.extract_text() """
    doc = model(text)
    ents_and_labels = [(ent.label_, ent.text) for ent in doc.ents if ent.label_ in custom_labels]
    print(ents_and_labels)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'c:\\Users\\jace\\Documents\\assignments\\dissertation\\fermentation_papers'